In [ ]:
# P1. Same source, eight noise seeds, every step inside each mechanism.
from pathlib import Path
import json,time,hashlib,copy,ast
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display,Markdown,HTML
assert 'D_PIPE' in globals() and 'D_CLEAN' in globals(), 'Attach to the idle Hands_Four_Methods_Computation_Dry_Run kernel.'
P_OUT=Path('results')/time.strftime('step_vectors_%Y%m%d_%H%M%S');P_OUT.mkdir(parents=True)
P_SEEDS=list(range(123100100,123100108));P_DEV=P_SEEDS[:4];P_TEST=P_SEEDS[4:]
P_METHODS=['ordinary','PAG','SoftPAG_h0_h1','SG_area_plus10','FreeU']
P_CONFIG={'source':str(D_SOURCE_PATHS[0]),'source_sha256':hashlib.sha256(D_SOURCE.tobytes()).hexdigest(),'seeds':P_SEEDS,'development_seeds':P_DEV,'heldout_seeds':P_TEST,'methods':P_METHODS,'schedule':100,'start':50,'end':100,'active_interventions':'every step 50..99','PAG_scale':1.0,'SoftPAG_heads':[0,1],'SoftPAG_u':.5,'SG_target':'reference attention area at same step times 1.1, preserve centroid','SG_delta_rms_fraction':.05,'FreeU':D_MANIFEST['FreeU'],'counterfactual_omission_steps':[50,70,90],'goal':'Describe and test step-wise contribution to each method effect across noise seeds. Endpoint direction is not a correctness target.'}
(P_OUT/'manifest.json').write_text(json.dumps(P_CONFIG,indent=2));D_SOURCE.save(P_OUT/'fixed_source.png')
# Freeze the exact previous runtime code for inspection and recovery, excluding credentials/API calls.
_pnb=json.loads(Path('Hands_Four_Methods_Computation_Dry_Run.ipynb').read_text())
_runtime=[]
for c in _pnb['cells']:
 s=''.join(c.get('source',[]))
 if any(s.startswith('# '+str(k)+'.') for k in [1,2,3,4,5]): _runtime.append(s)
(P_OUT/'runtime_snapshot.py').write_text((chr(10)*2).join(_runtime))
display(Markdown('''# Step vectors across noise seeds
One fixed source hand, eight re-noising seeds, all four methods plus ordinary denoising. Every method remains active throughout the 50-step continuation. Record each predicted clean latent, the direct intervention at the same state, and the accumulated change from earlier interventions. Decompose movement toward each final effect and test step omissions. All seeds and images stay in the record. Four development seeds are summarized before examining four held-out seeds; this is still one-image exploratory evidence.'''))
print('Output:',P_OUT,'GPU free GiB:',round(torch.cuda.mem_get_info()[0]/2**30,2))



In [ ]:
# P2. Trace the internal operations, direct forcing, and carried state separately.
P_RUNS={};P_REFS={};P_ROWS=[];P_COUNTER_ROWS=[]
def p_alpha(i): return D_SCHED.alphas_cumprod[D_SCHED.timesteps[i]].to('cuda')
def p_clean(z,eps,i):
 a=p_alpha(i);return (z-(1-a).sqrt()*eps)/a.sqrt()
def p_evaluate(z,i,method,ref_props=None,omit=False):
 with torch.no_grad():
  pair,tr=d_predict(z,i);base=d_cfg(pair);eps=base.clone()
  hp,hmask=d_properties(tr['cross_maps'][D_HAND].mean(0))
  att=tr['self']['attention'].float();ho=tr['self']['head_output'].float()
  info={'hand_area':float(hp[0]),'hand_centroid_x':float(hp[1]),'hand_centroid_y':float(hp[2]),'head_entropy':(-(att*att.clamp_min(1e-12).log()).sum(-1).mean(-1)).tolist(),'head_output_rms':ho.square().mean((1,2)).sqrt().tolist(),'head_identity_mass':att.diagonal(dim1=-2,dim2=-1).mean(-1).tolist(),'reference_forward_calls':1,'extra_forward_calls':0,'backward_calls':0}
  if not omit and method in ['PAG','SoftPAG_h0_h1']:
   heads=range(8) if method=='PAG' else [0,1];u=1. if method=='PAG' else .5
   pert,_=d_predict(z,i,heads,u);eps=base+(pair[1]-pert[1]);info['extra_forward_calls']=1
   info['mechanism']='eps_CFG + eps_cond - eps_cond_with_attention_interpolated_toward_identity'
  elif not omit and method=='FreeU':
   fp,ft=d_predict(z,i,freeu=True);eps=d_cfg(fp);info['extra_forward_calls']=1;info['freeu_features']=ft['freeu']
 if not omit and method=='SG_area_plus10':
  target=ref_props[i].to('cuda').clone();target[0]*=1.1
  g,sg=d_sg(z,i,target);delta=g*(.05*d_rms(base)/max(d_rms(g),1e-12));eps=(base.float()+delta).half()
  info.update(extra_forward_calls=1,backward_calls=1,sg_energy=sg['energy'],sg_gradient_rms=sg['gradient_rms'],sg_target=sg['target'],sg_properties=sg['properties'])
 with torch.no_grad():
  direct=p_clean(z,eps,i)-p_clean(z,base,i)
  info.update(local_delta_eps_rms=d_rms(eps-base),direct_clean_effect_rms=d_rms(direct),omitted=omit)
 return eps.detach(),p_clean(z,base,i).detach(),direct.detach(),hp.detach(),info
@torch.no_grad()
def p_start(seed):
 noise=torch.randn(D_CLEAN.shape,generator=torch.Generator(device='cuda').manual_seed(seed),device='cuda',dtype=D_CLEAN.dtype)
 return D_SCHED.add_noise(D_CLEAN,noise,D_SCHED.timesteps[50:51])
def p_run(seed,method):
 z=p_start(seed);clean_series=[];neutral_series=[];direct_series=[];infos=[];states={};props={};previous=D_CLEAN
 start=time.time()
 for i in range(50,100):
  states[i]=z.detach().cpu().clone()
  eps,neutral,direct,hp,info=p_evaluate(z,i,method,P_REFS.get(seed,{}).get('props'))
  with torch.no_grad():
   step=D_SCHED.step(eps,D_SCHED.timesteps[i],z,eta=0.);clean=step.pred_original_sample;z=step.prev_sample
   assert torch.isfinite(z).all()
   row={'seed':seed,'split':'development' if seed in P_DEV else 'heldout','method':method,'step':i,'timestep':int(D_SCHED.timesteps[i]),'clean_change_rms':d_rms(clean-previous),'eps_rms':d_rms(eps),'state_rms':d_rms(z),**{k:v for k,v in info.items() if isinstance(v,(int,float,bool,str))}}
   P_ROWS.append(row);previous=clean
   clean_series.append(clean.cpu());neutral_series.append(neutral.cpu());direct_series.append(direct.cpu());props[i]=hp.cpu();infos.append(info)
   if i in [50,54,59,64,69,74,79,84,89,94,99]:d_decode(clean).save(P_OUT/f's{seed}_{method}_step{i+1:03}.png')
 final=d_decode(z);final.save(P_OUT/f's{seed}_{method}_final.png')
 result={'seed':seed,'method':method,'clean':torch.cat(clean_series),'neutral':torch.cat(neutral_series),'direct':torch.cat(direct_series),'states':states,'props':props,'final':z.detach().cpu(),'internal_records':infos,'seconds_instrumented':time.time()-start}
 torch.save(result,P_OUT/f's{seed}_{method}_trace.pt');P_RUNS[seed,method]=result
 if method=='ordinary':P_REFS[seed]=result
 print('Finished',seed,method,'seconds',round(result['seconds_instrumented'],1),flush=True)
 return result
# Same-state neutral computation must match ordinary DDIM exactly.
with torch.no_grad():
 z=p_start(P_SEEDS[0]);eps,ne,di,hp,inf=p_evaluate(z,50,'ordinary')
 st=D_SCHED.step(eps,D_SCHED.timesteps[50],z,eta=0.)
 assert torch.equal(di,torch.zeros_like(di))
 assert float((ne-st.pred_original_sample).abs().max())==0
print('Direct/carried decomposition preflight passed. All existing model processors restored after every call.')



In [ ]:
# P3. Run the four development seeds first; all five paths are retained.
for seed in P_DEV:
 for method in P_METHODS: p_run(seed,method)
pd.DataFrame(P_ROWS).to_csv(P_OUT/'per_step_measurements.csv',index=False)
print('Development paths complete:',len(P_RUNS))
fig,axes=plt.subplots(len(P_DEV),5,figsize=(15,12))
for r,seed in enumerate(P_DEV):
 for c,method in enumerate(P_METHODS):
  axes[r,c].imshow(Image.open(P_OUT/f's{seed}_{method}_final.png'));axes[r,c].set_title(f'{seed}: {method}',fontsize=9);axes[r,c].axis('off')
plt.tight_layout();fig.savefig(P_OUT/'development_endpoints.png',dpi=130);plt.show()


In [ ]:
# P4. Attribute each step to the final vector, in a common clean-latent space.
# This is hindsight geometry. It is NOT an online reward, anatomical score, or causal proof.
P_ATTR=[];P_PATH_STATS=[];P_STAGE=[]
def p_dot(a,b): return float((a.double().flatten()*b.double().flatten()).sum())
def p_analyze(seeds):
 for seed in seeds:
  ref=P_REFS[seed]
  for method in P_METHODS[1:]:
   r=P_RUNS[seed,method];effects=r['clean'].float()-ref['clean'].float();end=(r['final']-ref['final']).float()
   denom=p_dot(end,end);assert denom>1e-12
   seq=torch.cat([torch.zeros_like(effects[:1]),effects,end],dim=0);increments=seq[1:]-seq[:-1]
   fractions=np.array([p_dot(v,end)/denom for v in increments]);assert abs(fractions.sum()-1)<1e-5
   cum=np.cumsum(fractions);final_norm=denom**.5
   own_end=r['final'].float()-D_CLEAN.cpu().float();own_denom=p_dot(own_end,own_end)
   own_seq=torch.cat([D_CLEAN.cpu().float(),r['clean'].float(),r['final'].float()],dim=0);own_inc=own_seq[1:]-own_seq[:-1]
   own_fractions=[p_dot(v,own_end)/max(own_denom,1e-12) for v in own_inc]
   for k,step in enumerate(list(range(50,100))+[100]):
    e=seq[k+1];proj=cum[k]*end[0];orth=d_rms(e-proj)*e.numel()**.5/final_norm
    direct=r['direct'][k].float() if k<50 else torch.zeros_like(end[0]);carried=(r['neutral'][k].float()-ref['clean'][k].float()) if k<50 else end[0]
    row={'seed':seed,'split':'development' if seed in P_DEV else 'heldout','method':method,'step':step,'kind':'first clean forecast' if k==0 else ('final readout' if step==100 else 'next clean forecast'),'signed_final_effect_fraction':float(fractions[k]),'cumulative_final_effect_fraction':float(cum[k]),'orthogonal_effect_norm_ratio':orth,'increment_rms':d_rms(increments[k]),'direct_projection':p_dot(direct,end)/denom,'carried_projection':p_dot(carried,end)/denom,'own_source_to_end_fraction':own_fractions[k]}
    P_ATTR.append(row)
   stats={'seed':seed,'split':'development' if seed in P_DEV else 'heldout','method':method,'final_effect_rms':d_rms(end),'path_efficiency':final_norm/sum(float(v.double().norm()) for v in increments),'negative_fraction_count':int((fractions<0).sum()),'total_backward_projection':float(-fractions[fractions<0].sum()),'max_overshoot':float(cum.max()),'max_orthogonal_excursion':max(x['orthogonal_effect_norm_ratio'] for x in P_ATTR if x['seed']==seed and x['method']==method)}
   P_PATH_STATS.append(stats)
   for start in [50,60,70,80,90]:
    vals=fractions[start-50:start-40];P_STAGE.append({'seed':seed,'split':stats['split'],'method':method,'stage':f'{start}-{start+9}','signed_fraction':float(vals.sum())})
 pd.DataFrame(P_ATTR).to_csv(P_OUT/'step_vector_attribution.csv',index=False)
 pd.DataFrame(P_PATH_STATS).to_csv(P_OUT/'path_statistics.csv',index=False)
 pd.DataFrame(P_STAGE).to_csv(P_OUT/'stage_contributions.csv',index=False)
p_analyze(P_DEV)
P_DEV_STAGE=pd.DataFrame(P_STAGE).groupby(['method','stage']).signed_fraction.agg(['mean','std','min','max'])
P_DEV_STAGE.to_csv(P_OUT/'development_stage_patterns.csv');display(P_DEV_STAGE.round(4))
fig,axes=plt.subplots(4,3,figsize=(17,13))
for row,method in enumerate(P_METHODS[1:]):
 for seed in P_DEV:
  df=pd.DataFrame([r for r in P_ATTR if r['seed']==seed and r['method']==method])
  axes[row,0].plot(df.step,df.cumulative_final_effect_fraction,label=str(seed)[-3:])
  axes[row,1].plot(df.step,df.signed_final_effect_fraction)
  axes[row,2].plot(df.step,df.orthogonal_effect_norm_ratio)
 axes[row,0].axhline(1,color='black',ls='--');axes[row,1].axhline(0,color='black',lw=.7)
 for col,label in enumerate(['accumulated projection onto final effect','signed contribution of this step','orthogonal excursion / final-effect norm']):
  axes[row,col].set_title(method+': '+label,fontsize=10);axes[row,col].set_xlabel('step')
 axes[row,0].legend(title='noise seed suffix')
plt.tight_layout();fig.savefig(P_OUT/'development_vector_paths.png',dpi=130);plt.show()
print('Signed fractions telescope to 1. Negative means movement away from the final effect, not a bad image. Early points are clean-image forecasts, not decoded noisy latents.')


In [ ]:
# P5. Causal check: omit ONE intervention, preserve its state and every later operation.
# A step important for reproducing this endpoint need not be important for image quality.
def p_continue(seed,method,start,omit_step=None):
 z=P_RUNS[seed,method]['states'][start].to('cuda').clone()
 for i in range(start,100):
  eps,_,_,_,_=p_evaluate(z,i,method,P_REFS[seed]['props'],omit=(i==omit_step))
  with torch.no_grad():z=D_SCHED.step(eps,D_SCHED.timesteps[i],z,eta=0.).prev_sample
 return z.detach().cpu()
def p_omit_tests(seeds):
 for seed in seeds:
  ref=P_REFS[seed]['final'].float()
  for method in P_METHODS[1:]:
   full=P_RUNS[seed,method]['final'].float();effect=full-ref;denom=p_dot(effect,effect)
   for omitted in [50,70,90]:
    cf=p_continue(seed,method,omitted,omitted).float();delta=full-cf
    row={'seed':seed,'split':'development' if seed in P_DEV else 'heldout','method':method,'omitted_step':omitted,'endpoint_change_rms':d_rms(delta),'fraction_final_effect_lost':p_dot(delta,effect)/denom,'retained_projection':p_dot(cf-ref,effect)/denom,'off_axis_change_ratio':float((delta-(p_dot(delta,effect)/denom)*effect).norm())/(denom**.5),'observed_step_fraction':next(x['signed_final_effect_fraction'] for x in P_ATTR if x['seed']==seed and x['method']==method and x['step']==omitted)}
    P_COUNTER_ROWS.append(row);d_decode(cf.half().to('cuda')).save(P_OUT/f's{seed}_{method}_omit{omitted}_final.png')
    torch.save(cf,P_OUT/f's{seed}_{method}_omit{omitted}_final.pt')
   print('Omission tests finished',seed,method,flush=True)
  pd.DataFrame(P_COUNTER_ROWS).to_csv(P_OUT/'single_step_omissions.csv',index=False)
repeat=p_continue(P_DEV[0],'PAG',70,None)
repeat_error=float((repeat-P_RUNS[P_DEV[0],'PAG']['final']).abs().max());assert repeat_error==0,repeat_error
print('Same-prefix replay matches exactly:',repeat_error,flush=True)
p_omit_tests(P_DEV)
display(pd.DataFrame(P_COUNTER_ROWS).groupby(['method','omitted_step'])[['fraction_final_effect_lost','endpoint_change_rms']].agg(['mean','std']).round(5))
print('These are single-step omissions at 50, 70 and 90, not a claim that every remaining step is necessary.')


In [ ]:
# P6. Freeze development observations, then evaluate four untouched noise seeds.
P_HYPOTHESES={
 'FreeU_early_forecast':'The first ten clean forecasts account for the largest positive projected stage contribution.',
 'PAG_seed_dependence':'PAG has more variation across seeds in the early projected contribution than FreeU.',
 'SG_reversals':'Normalized self-guidance shows more backward projected motion than FreeU; a local target gradient need not give a smooth global path.',
 'causal_caution':'A large forecast projection can coexist with a small effect from omitting that step. Omission tests, not projections alone, determine endpoint sensitivity.',
 'scope':'These claims concern the chosen four controls and this source image. They do not assert better anatomy or faster generation.'}
(P_OUT/'frozen_development_hypotheses.json').write_text(json.dumps(P_HYPOTHESES,indent=2))
print('Development hypotheses frozen before held-out paths.',flush=True)
for seed in P_TEST:
 for method in P_METHODS:p_run(seed,method)
p_analyze(P_TEST)
p_omit_tests(P_TEST)
pd.DataFrame(P_ROWS).to_csv(P_OUT/'per_step_measurements.csv',index=False)
print('All seed trajectories and omission continuations complete.',flush=True)
fig,axes=plt.subplots(len(P_TEST),5,figsize=(15,12))
for r,seed in enumerate(P_TEST):
 for c,method in enumerate(P_METHODS):
  axes[r,c].imshow(Image.open(P_OUT/f's{seed}_{method}_final.png'));axes[r,c].set_title(f'{seed}: {method}',fontsize=9);axes[r,c].axis('off')
plt.tight_layout();fig.savefig(P_OUT/'heldout_endpoints.png',dpi=130);plt.show()
display(pd.DataFrame(P_STAGE).groupby(['split','method','stage']).signed_fraction.agg(['mean','std']).round(4))


In [ ]:
# P7. Expose the exact attention transformation at every stored step.
# Replays are diagnostic overhead, counted separately; they never alter a trajectory.
P_ATTN_INTERNAL=[]
for seed in P_SEEDS:
 for method in ['PAG','SoftPAG_h0_h1']:
  r=P_RUNS[seed,method];heads=list(range(8)) if method=='PAG' else [0,1];u=1. if method=='PAG' else .5
  for i in range(50,100):
   with torch.no_grad():
    _,tr=d_predict(r['states'][i].to('cuda'),i);cap=tr['self'];a=cap['attention'].float();o=cap['head_output'].float();v=cap['v'].float()
    changed=a.clone();changed[heads]=(1-u)*a[heads]+u*torch.eye(a.shape[-1])[None]
    changed_o=o.clone();changed_o[heads]=(1-u)*o[heads]+u*v[heads]
    entropy=-(a*a.clamp_min(1e-12).log()).sum(-1).mean(-1)
    after_entropy=-(changed*changed.clamp_min(1e-12).log()).sum(-1).mean(-1)
    for h in range(8):
     P_ATTN_INTERNAL.append({'seed':seed,'method':method,'step':i,'head':h,'selected':h in heads,'identity_interpolation':u if h in heads else 0.,'entropy_before':float(entropy[h]),'entropy_after':float(after_entropy[h]),'head_output_rms_before':d_rms(o[h]),'head_output_rms_after':d_rms(changed_o[h]),'head_output_delta_rms':d_rms(changed_o[h]-o[h]),'q_rms':d_rms(cap['q'][h]),'k_rms':d_rms(cap['k'][h]),'v_rms':d_rms(v[h])})
  print('Internal attention replay',seed,method,flush=True)
pd.DataFrame(P_ATTN_INTERNAL).to_csv(P_OUT/'per_step_attention_transformations.csv',index=False)
# Self-guidance target errors and actual FreeU stages are already retained in every trace.
P_SG_INTERNAL=[];P_FREEU_INTERNAL=[]
for seed in P_SEEDS:
 for k,info in enumerate(P_RUNS[seed,'SG_area_plus10']['internal_records']):
  P_SG_INTERNAL.append({'seed':seed,'step':k+50,'energy':info['sg_energy'],'gradient_rms':info['sg_gradient_rms'],'area':info['sg_properties'][0],'target_area':info['sg_target'][0],'signed_area_error':info['sg_properties'][0]-info['sg_target'][0],'centroid_error':float(np.linalg.norm(np.array(info['sg_properties'][1:])-np.array(info['sg_target'][1:])))})
 for k,info in enumerate(P_RUNS[seed,'FreeU']['internal_records']):
  for stage in info['freeu_features']:P_FREEU_INTERNAL.append({'seed':seed,'step':k+50,**stage})
pd.DataFrame(P_SG_INTERNAL).to_csv(P_OUT/'per_step_self_guidance_gradients.csv',index=False)
pd.DataFrame(P_FREEU_INTERNAL).to_csv(P_OUT/'per_step_freeu_features.csv',index=False)
print('Saved all-step attention inputs/outputs, gradient/target traces and FreeU feature changes.')


In [ ]:
# P8. Compare internal signatures, hindsight movement, and actual omission sensitivity.
assert len(P_RUNS)==40 and len(P_COUNTER_ROWS)==96
A=pd.DataFrame(P_ATTR);S=pd.DataFrame(P_STAGE);C=pd.DataFrame(P_COUNTER_ROWS);T=pd.DataFrame(P_PATH_STATS);G=pd.DataFrame(P_SG_INTERNAL)
P_STAGE_SUMMARY=S.groupby(['split','method','stage']).signed_fraction.agg(['mean','std','min','max'])
P_CAUSAL_SUMMARY=C.groupby(['split','method','omitted_step']).fraction_final_effect_lost.agg(['mean','std','min','max'])
P_STAGE_SUMMARY.to_csv(P_OUT/'stage_patterns_by_split.csv');P_CAUSAL_SUMMARY.to_csv(P_OUT/'omission_patterns_by_split.csv')
P_OSCILLATIONS=[]
for seed in P_SEEDS:
 g=G[G.seed==seed].sort_values('step');err=g.signed_area_error.to_numpy()
 P_OSCILLATIONS.append({'seed':seed,'split':'development' if seed in P_DEV else 'heldout','target_error_sign_changes':int(np.sum(np.sign(err[1:])!=np.sign(err[:-1]))),'mean_relative_area_error':float(np.mean(np.abs(err)/g.target_area.to_numpy())),'max_relative_area_error':float(np.max(np.abs(err)/g.target_area.to_numpy()))})
display(T.groupby(['split','method'])[['final_effect_rms','total_backward_projection','path_efficiency','max_orthogonal_excursion']].mean().round(4))
display(P_CAUSAL_SUMMARY.round(4));display(pd.DataFrame(P_OSCILLATIONS))
fig,axes=plt.subplots(4,3,figsize=(17,14))
for row,method in enumerate(P_METHODS[1:]):
 for seed in P_SEEDS:
  d=A[(A.seed==seed)&(A.method==method)];ls='-' if seed in P_TEST else '--';alpha=1. if seed in P_TEST else .45
  axes[row,0].plot(d.step,d.cumulative_final_effect_fraction,ls=ls,alpha=alpha,label=str(seed)[-3:])
  axes[row,1].plot(d.step,d.orthogonal_effect_norm_ratio,ls=ls,alpha=alpha)
 cf=C[C.method==method]
 for seed in P_SEEDS:
  d=cf[cf.seed==seed];axes[row,2].plot(d.omitted_step,100*d.fraction_final_effect_lost,'o-',alpha=.7,label=str(seed)[-3:])
 axes[row,0].axhline(1,color='black',ls=':');axes[row,2].axhline(0,color='black',lw=.7)
 axes[row,0].set_title(method+': projected final effect');axes[row,1].set_title('Off-direction motion / final-effect norm');axes[row,2].set_title('Final effect lost when ONE step omitted (%)')
 for ax in axes[row]:ax.set_xlabel('step')
 axes[row,0].legend(ncol=4,fontsize=7)
plt.tight_layout();fig.savefig(P_OUT/'all_seed_paths_and_causal_checks.png',dpi=140);plt.show()
fig,axes=plt.subplots(2,2,figsize=(15,9))
for seed in P_SEEDS:
 d=G[G.seed==seed];axes[0,0].plot(d.step,d.signed_area_error/d.target_area,label=str(seed)[-3:]);axes[0,1].plot(d.step,d.energy)
 for method,ax in [('PAG',axes[1,0]),('FreeU',axes[1,1])]:
  d=A[(A.seed==seed)&(A.method==method)&(A.step<100)];ax.plot(d.step,d.direct_projection,alpha=.7);ax.plot(d.step,d.carried_projection,ls='--',alpha=.7)
axes[0,0].set_title('Self-guidance: relative signed target-area error');axes[0,0].axhline(0,color='black');axes[0,0].legend(ncol=4,fontsize=8)
axes[0,1].set_title('Self-guidance: actual energy before each gradient')
axes[1,0].set_title('PAG: direct (solid) versus carried (dashed) projection')
axes[1,1].set_title('FreeU: direct (solid) versus carried (dashed) projection')
plt.tight_layout();fig.savefig(P_OUT/'internal_mechanism_dynamics.png',dpi=140);plt.show()
P_ASSOCIATIONS=[]
for split in ['development','heldout']:
 for method in P_METHODS[1:]:
  d=C[(C.split==split)&(C.method==method)]
  P_ASSOCIATIONS.append({'split':split,'method':method,'n':len(d),'spearman_forecast_credit_vs_omission':d.observed_step_fraction.corr(d.fraction_final_effect_lost,method='spearman')})
display(pd.DataFrame(P_ASSOCIATIONS));pd.DataFrame(P_OSCILLATIONS).to_csv(P_OUT/'self_guidance_oscillations.csv',index=False)
pd.DataFrame(P_ASSOCIATIONS).to_csv(P_OUT/'projection_vs_causality.csv',index=False)
print('A small or negative omission effect contradicts treating that step forecast as indispensable. No anatomy labels inferred from tensors.')


In [ ]:
# P9. Compact, auditable results for all seeds and all omission tests.
print('OUTPUT_DIRECTORY',P_OUT)
print('COUNTS',len(P_RUNS),len(P_COUNTER_ROWS),len(P_ATTN_INTERNAL))
print('HELDOUT STAGES\n',P_STAGE_SUMMARY.loc['heldout'].round(4).to_string())
print('OMISSION RESULTS\n',P_CAUSAL_SUMMARY.round(4).to_string())
print('PATH STATISTICS\n',T.groupby(['split','method'])[['total_backward_projection','path_efficiency','max_orthogonal_excursion']].mean().round(4).to_string())
print('SG OSCILLATIONS\n',pd.DataFrame(P_OSCILLATIONS).round(4).to_string(index=False))
print('ASSOCIATIONS\n',pd.DataFrame(P_ASSOCIATIONS).round(4).to_string(index=False))
print('FREEU COLUMNS',list(pd.DataFrame(P_FREEU_INTERNAL).columns))


In [ ]:
# P10. Decode every saved clean forecast for a step-by-step inspector (no new U-Net evaluations).
import io,base64,html
P_PREVIEWS={}
def p_uri(im,fmt='JPEG',quality=82):
    buf=io.BytesIO();im.save(buf,format=fmt,quality=quality)
    return 'data:image/'+fmt.lower()+';base64,'+base64.b64encode(buf.getvalue()).decode()
for seed in P_SEEDS:
    for method in P_METHODS:
        r=P_RUNS[seed,method];frames=[]
        for k in range(50):
            im=d_decode(r['clean'][k:k+1].to('cuda')).resize((192,192),Image.Resampling.LANCZOS)
            frames.append(p_uri(im))
        P_PREVIEWS[str(seed)+'|'+method]=frames
    print('All 50 forecast previews decoded:',seed,flush=True)
print('ATTRIBUTION_FIELDS',list(A.columns))
print('Preview count',sum(map(len,P_PREVIEWS.values())))


In [ ]:
# P11. Self-contained inspector: every seed, step, mechanism and causal probe.
P_PACKET={'seeds':P_SEEDS,'methods':P_METHODS[1:],'frames':P_PREVIEWS,'rows':json.loads(A.to_json(orient='records')),'omissions':json.loads(C.to_json(orient='records')),'internals':{},'attention':json.loads(pd.DataFrame(P_ATTN_INTERNAL).to_json(orient='records')),'finals':{}}
for seed in P_SEEDS:
    for method in P_METHODS:
        key=str(seed)+'|'+method
        P_PACKET['finals'][key]=p_uri(Image.open(P_OUT/f's{seed}_{method}_final.png').resize((256,256)))
        if method!='ordinary':P_PACKET['internals'][key]=P_RUNS[seed,method]['internal_records']
P_REPORT_TEMPLATE=r'''<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Inside each diffusion step</title><style>
body{font:16px/1.5 system-ui;margin:0;background:#f5f7fb;color:#182338}main{max-width:1250px;margin:auto;padding:26px}h1{font-size:32px;margin-bottom:8px}h2{margin-top:30px}p{max-width:950px}.card{background:white;border:1px solid #d4deea;border-radius:12px;padding:18px;margin:20px 0}.controls{display:flex;gap:20px;flex-wrap:wrap}select,input{font:inherit}select{padding:7px}input[type=range]{width:100%}.images{display:grid;grid-template-columns:repeat(3,1fr);gap:12px}.images img{width:100%;max-width:300px;border-radius:7px}figure{margin:0}figcaption{font-size:14px}pre{white-space:pre-wrap;word-break:break-word;font-size:12px;max-height:450px;overflow:auto;background:#f1f4f9;padding:12px}.plot{width:100%;height:auto}.scroll{overflow:auto}table{border-collapse:collapse;font-size:13px}td,th{padding:7px;border-bottom:1px solid #ddd;text-align:right}.note{border-left:4px solid #5275c7;padding-left:15px}.stats{font-size:14px}@media(max-width:650px){main{padding:14px}.images{grid-template-columns:repeat(2,1fr)}}
</style></head><body><main><h1>Inside each diffusion step</h1><p>One fixed source hand · eight predeclared re-noising seeds · four mechanisms · fifty continuation steps. Four development seeds and four held-out seeds; every result retained.</p>
<p class="note"><b>What the vector means:</b> movement toward a method's eventual effect relative to ordinary diffusion, in predicted-clean latent coordinates. This is a hindsight direction, not an anatomical-quality score. The source-to-own-end vector is also in the step values.</p>
<div class="card"><h2>Inspect an individual computation</h2><div class="controls"><label>Seed <select id="seed"></select></label><label>Mechanism <select id="method"></select></label></div><p id="stepLabel"></p><input aria-label="Continuation step" id="step" type="range" min="0" max="49" value="0"><div class="images"><figure><img id="source" src="__SOURCE__"><figcaption>Fixed source: original seed 123</figcaption></figure><figure><img id="ordinary"><figcaption>Ordinary clean forecast at this step</figcaption></figure><figure><img id="current"><figcaption>Method clean forecast at this step</figcaption></figure></div><p id="stats" class="stats"></p><div class="images"><figure><img id="baselineFinal"><figcaption>Ordinary endpoint</figcaption></figure><figure><img id="methodFinal"><figcaption>Method endpoint</figcaption></figure></div><details open><summary>Mechanism and exact step values</summary><p id="mechanism"></p><pre id="values"></pre></details><details><summary>Same-prefix omission test at this step, if tested</summary><pre id="causal"></pre></details></div>
<div class="card"><h2>Actual architectural changes</h2><ul><li><b>PAG:</b> replace all eight mid-block 8×8 self-attention matrices by identity in a perturbed conditional branch. Add the ordinary-minus-perturbed prediction to CFG, scale 1.</li><li><b>SoftPAG adaptation:</b> interpolate heads 0 and 1 halfway from attention to identity at the same layer. This SD1.5 adaptation is not a reproduction on another architecture.</li><li><b>Self-guidance adaptation:</b> differentiate a soft hand-token attention-area/centroid energy at one 16×16 cross-attention layer. Area target is 1.1× the ordinary reference at that step. Normalize the gradient contribution to 5% of ordinary prediction RMS. This requires an ordinary reference run.</li><li><b>FreeU:</b> actual decoder backbone/skip feature scaling, b1=1.2, b2=1.4, s1=0.9, s2=0.2; measured feature changes retained.</li></ul></div>
<div class="card"><h2>All seeds: paths and causal checks</h2><img class="plot" src="__PATHS__"><p>Dashed: development. Solid: held out. Each omission removes the intervention at one step and completes the trajectory. It retains the diffusion update. Fixed probes: indices 50, 70, 90 for every method and seed; 96 counterfactuals.</p></div>
<div class="card"><h2>Internal dynamics</h2><img class="plot" src="__INTERNAL__"><p>Direct projection isolates the intervention at the same state. Carried projection includes inherited changes. Soft attention area is not anatomical hand area.</p></div>
<div class="card"><h2>Every endpoint</h2><img class="plot" src="__DEV__"><img class="plot" src="__TEST__"></div>
<div class="card"><h2>Numerical evidence</h2><h3>Stage contributions</h3><div class="scroll">__STAGES__</div><h3>Endpoint effect lost by omitting one intervention</h3><div class="scroll">__CAUSAL__</div><h3>Forecast credit versus omission sensitivity</h3><div class="scroll">__ASSOCIATIONS__</div><h3>Self-guidance error sign changes</h3><div class="scroll">__OSCILLATIONS__</div></div>
<div class="card"><h2>Definitions and limits</h2><p>eᵢ = clean_methodᵢ − clean_ordinaryᵢ; v = final_method − final_ordinary; step credit = ⟨eᵢ − eᵢ₋₁,v⟩/⟨v,v⟩. The first value is a clean forecast from the shared noisy start, not a completed latent displacement. A terminal correction makes credits sum to one. Direct = ⟨clean_intervened − clean_neutral_at_same_state,v⟩/⟨v,v⟩. Orthogonal excursion retains movement unexplained by this direction.</p><p>Omission effect = ⟨final_full − final_omitted,v⟩/⟨v,v⟩. Zero may mean redundancy or compensation; negative does not mean better anatomy. Individual omissions do not establish what a whole window or interacting set of steps does.</p><p>Eight noise seeds of one image do not establish transfer across hands or prompts. The final direction is unavailable online. No Jev controller, anatomy judge or speedup is tested here. Reference runs, 800 diagnostic attention replays and counterfactuals are research overhead. Model weights, prompt and schedule remain fixed.</p><p>The report folder contains latent/state traces, CSVs, frozen hypotheses, configuration and runtime snapshot. Notebook: Hands_Step_Vector_Attribution_Across_Seeds.ipynb.</p></div>
<script>const data=__DATA__;const E=id=>document.getElementById(id);data.seeds.forEach((s,i)=>E('seed').add(new Option(s+(i<4?' · development':' · held out'),s)));data.methods.forEach(m=>E('method').add(new Option(m,m)));const explanations={PAG:'All eight attention matrices become identity in the perturbed branch; the prediction difference changes the denoiser output.',SoftPAG_h0_h1:'Heads 0 and 1: A → 0.5A + 0.5I, hence AV → 0.5AV + 0.5V in the perturbed branch.',SG_area_plus10:'Differentiate area/centroid energy, normalize its prediction contribution, then make the DDIM update.',FreeU:'Scale decoder backbone features and Fourier-filter skip features at the configured stages.'};function draw(){const s=Number(E('seed').value),m=E('method').value,k=Number(E('step').value),step=k+50,key=s+'|'+m;const row=data.rows.find(r=>r.seed===s&&r.method===m&&r.step===step);E('stepLabel').textContent='Continuation '+(k+1)+' / 50 · schedule index '+step+' · clean forecast before its update';E('ordinary').src=data.frames[s+'|ordinary'][k];E('current').src=data.frames[key][k];E('baselineFinal').src=data.finals[s+'|ordinary'];E('methodFinal').src=data.finals[key];E('mechanism').textContent=explanations[m];E('stats').textContent='Projected effect: '+(100*row.cumulative_final_effect_fraction).toFixed(1)+'% · direct: '+(100*row.direct_projection).toFixed(1)+'% · carried: '+(100*row.carried_projection).toFixed(1)+'% · off-direction norm: '+row.orthogonal_effect_norm_ratio.toFixed(3)+' × endpoint-effect norm';E('values').textContent=JSON.stringify({attribution:row,internal_measurements:data.internals[key][k],attention_heads:data.attention.filter(r=>r.seed===s&&r.method===m&&r.step===step)},null,2);const cf=data.omissions.filter(r=>r.seed===s&&r.method===m&&r.omitted_step===step);E('causal').textContent=cf.length?JSON.stringify(cf,null,2):'Not tested here. Fixed omission probes are indices 50, 70 and 90.';}['seed','method','step'].forEach(id=>E(id).addEventListener('input',draw));draw();</script></main></body></html>'''
def p_plot_uri(filename):return p_uri(Image.open(P_OUT/filename).convert('RGB'),quality=90)
replacements={'__SOURCE__':p_uri(Image.open(P_OUT/'fixed_source.png').resize((256,256))),'__PATHS__':p_plot_uri('all_seed_paths_and_causal_checks.png'),'__INTERNAL__':p_plot_uri('internal_mechanism_dynamics.png'),'__DEV__':p_plot_uri('development_endpoints.png'),'__TEST__':p_plot_uri('heldout_endpoints.png'),'__STAGES__':P_STAGE_SUMMARY.round(4).to_html(),'__CAUSAL__':P_CAUSAL_SUMMARY.round(4).to_html(),'__ASSOCIATIONS__':pd.DataFrame(P_ASSOCIATIONS).round(4).to_html(index=False),'__OSCILLATIONS__':pd.DataFrame(P_OSCILLATIONS).round(4).to_html(index=False),'__DATA__':json.dumps(P_PACKET,default=lambda x:x.tolist() if hasattr(x,'tolist') else str(x))}
P_REPORT=P_REPORT_TEMPLATE
for key,value in replacements.items():P_REPORT=P_REPORT.replace(key,value)
(P_OUT/'report.html').write_text(P_REPORT)
P_REPORT_URL='/files/workspace/crazy_exp/'+str(P_OUT)+'/report.html'
display(HTML('<h3>Completed: 40 trajectories, 96 omission tests, 2,000 clean-image forecasts.</h3><a target="_blank" href="'+P_REPORT_URL+'">Open the step-by-step computation inspector</a>'))
print('Report MB',round((P_OUT/'report.html').stat().st_size/1e6,2))


In [ ]:
# P12. Source-to-own-end direction, findings, and reproducibility notes.
P_OWN_ROWS=[]
for seed in P_SEEDS:
    for method in P_METHODS:
        r=P_RUNS[seed,method];v=r['final'].double()-D_CLEAN.detach().cpu().double();den=float((v*v).sum())
        for k in range(50):
            shift=r['clean'][k:k+1].double()-D_CLEAN.detach().cpu().double()
            P_OWN_ROWS.append({'seed':seed,'method':method,'step':k+50,'source_to_own_end_projection':float((shift*v).sum())/den})
pd.DataFrame(P_OWN_ROWS).to_csv(P_OUT/'source_to_own_end_projection.csv',index=False)
fig,ax=plt.subplots(figsize=(12,5))
for method in P_METHODS:
    d=pd.DataFrame(P_OWN_ROWS);d=d[d.method==method].groupby('step').source_to_own_end_projection.agg(['mean','min','max'])
    ax.plot(d.index,d['mean'],label=method);ax.fill_between(d.index,d['min'],d['max'],alpha=.08)
ax.axhline(1,color='black',ls=':');ax.set(xlabel='Schedule index',ylabel='Projection onto source-to-own-end direction',title='Every method: clean-image forecasts along its source-to-final vector (mean and seed range)');ax.legend();plt.tight_layout();fig.savefig(P_OUT/'source_to_own_end.png',dpi=140);plt.show()
P_FINDINGS='''# What this run supports

We traced eight fixed re-noising seeds of the original seed-123 source image, all fifty continuation steps, four mechanisms plus ordinary diffusion, and 96 same-prefix single-intervention omissions. No seed or outcome was discarded. Development hypotheses were saved before the held-out runs.

- FreeU has a repeatable early forecast signature: its first ten forecasts account for 66.75% of the final-effect projection in development and 63.38% held out. Omitting the first intervention loses 8.15% of that projection on held-out seeds, versus 1.58% at index 70 and 0.56% at 90. Forecast magnitude and causal contribution are different quantities.
- PAG's early-stage mean repeats (50.18% development, 49.83% held out), but varies more by seed than FreeU. Its one-step omission losses on held-out seeds are 3.50%, 1.30%, 0.47% at indices 50,70,90.
- SoftPAG's middle-stage behavior is seed-dependent: the mean 70-79 contribution is +35.48% development but -10.97% held out. A fixed interpretation such as 'these heads always help in this phase' is unsupported.
- The normalized self-guidance adaptation repeatedly crosses its changing attention-area target: 40-46 sign changes in 49 transitions across all eight seeds. This is a concrete candidate for an adaptive-gain or damping experiment, not proof of its cause. Fixed gradient normalization and moving reference targets both need investigation.
- Hindsight step credit correlates with omission sensitivity on held-out data (Spearman: FreeU 0.888, PAG 0.594, SoftPAG 0.601, self-guidance 0.476; 12 probes per method). These pooled correlations are small-sample and confounded by timestep; they do not establish that Jev can predict useful changes online.

This gives the proposed approach a measurable foothold: recurring internal dynamics, different roles across phases, and testable consequences of interventions. It does not demonstrate improved anatomy, a faster route, or Jev's added value. A useful next controller would receive current error, recent error signs, prediction changes, head/feature measurements and prior intervention outcomes, and choose bounded joint controls. It must not receive the future endpoint vector for a held-out run. Compare it with a simple feedback rule under the same compute allowance before assigning gains to Jev's reasoning.

The central goal remains better images through better inference paths. A geometric final-effect direction explains how a method changes an image; an independent image-quality assessment is still needed to call that change better.
'''
(P_OUT/'findings.md').write_text(P_FINDINGS)
display(Markdown(P_FINDINGS))
# Insert the additional start-to-end plot and findings into the standalone report.
card='<div class="card"><h2>Source to each method endpoint</h2><img class="plot" src="'+p_plot_uri('source_to_own_end.png')+'"><p>Mean and full seed range. This includes ordinary diffusion. Later effect-relative plots isolate what each intervention changes beyond the ordinary trajectory.</p></div>'
P_REPORT=P_REPORT.replace('<div class="card"><h2>All seeds: paths and causal checks</h2>',card+'<div class="card"><h2>All seeds: paths and causal checks</h2>')
P_REPORT=P_REPORT.replace('<div class="card"><h2>Definitions and limits</h2>','<div class="card"><h2>Interpretation</h2><pre>'+html.escape(P_FINDINGS)+'</pre></div><div class="card"><h2>Definitions and limits</h2>')
(P_OUT/'report.html').write_text(P_REPORT)
(P_OUT/'README.txt').write_text('Notebook: Hands_Step_Vector_Attribution_Across_Seeds.ipynb\nThis notebook shared the existing dry-run kernel/model to avoid a second GPU allocation. On a fresh kernel, review and execute runtime_snapshot.py before P1; it captures the model/bootstrap helpers from the prior dry-run notebook. To analyze without rerunning generation, load s*_trace.pt and the CSVs from this folder. The report is self-contained and uses embedded images. The instrumentation is research overhead, not a speed benchmark.\nAll projections are in unweighted VAE latent coordinates, not a perceptual or anatomical metric. Only single interventions at three fixed indices were omitted.\n')
assert len(P_RUNS)==40 and len(P_COUNTER_ROWS)==96 and sum(map(len,P_PREVIEWS.values()))==2000
assert all(torch.isfinite(r['clean']).all() and torch.isfinite(r['final']).all() for r in P_RUNS.values())
print('Verified: all 40 paths finite; all 96 causal probes retained; all 2000 previews available.')
display(HTML('<a target="_blank" href="'+P_REPORT_URL+'">Open the complete computation inspector and findings</a>'))


In [ ]:
# Browser reconnection check — no model or experiment state changes.
import datetime
print('Cell execution OK:', datetime.datetime.now().isoformat(timespec='seconds'))
print('Diffusion pipeline loaded:', 'D_PIPE' in globals())
print('Original source latent retained:', 'D_CLEAN' in globals())
print('Prior trajectories retained:', len(globals().get('P_RUNS', {})))
if 'torch' in globals() and torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Free GPU GiB:', round(torch.cuda.mem_get_info()[0] / 2**30, 2))